In [ ]:
# -----------------------------------------------------------
# Enhanced CS-340 Dashboard
# Includes: Logging + Error Handling Enhancements
# Created by: Shawn Randall
# -----------------------------------------------------------

# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# OS + Data
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Logging
import logging
logging.basicConfig(
    filename='dashboard.log',
    level=logging.INFO,
    format='%(asctime)s:%(levelname)s:%(message)s'
)
logging.info("Dashboard started successfully.")

# CRUD module
from crud import AnimalShelter

# -----------------------------------------------------------
# Database Connection
# -----------------------------------------------------------
username = "aacuser"
password = "MySecurePass123!"

try:
    db = AnimalShelter(username, password)
    logging.info("Connected to MongoDB successfully.")
except Exception as e:
    logging.error(f"Database connection failed: {e}", exc_info=True)

# Load initial dataset
try:
    df = pd.DataFrame.from_records(db.read({}))
    df.drop(columns=['_id'], inplace=True, errors='ignore')
    logging.info(f"Initial dataset loaded with {len(df)} records.")
except Exception as e:
    logging.error(f"Failed to load initial dataset: {e}", exc_info=True)
    df = pd.DataFrame()
    

# -----------------------------------------------------------
# Logo
# -----------------------------------------------------------
image_filename = 'Grazioso Salvare Logo.png'
try:
    encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()
except Exception as e:
    logging.error(f"Logo file missing or unreadable: {e}", exc_info=True)
    encoded_image = None


# -----------------------------------------------------------
# Dashboard Layout
# -----------------------------------------------------------
app = JupyterDash(__name__)

app.layout = html.Div([
    # Logo
    html.Img(
        src=f'data:image/png;base64,{encoded_image}' if encoded_image else "",
        style={'height': '100px', 'width': 'auto'}
    ),

    # Header
    html.Div([
        html.A(
            html.Img(
                src=f'data:image/png;base64,{encoded_image}' if encoded_image else "",
                style={'height': '60px'}
            ),
            href="http://www.snhu.edu", target="_blank"
        ),
        html.Div("Created by Shawn Randall - Enhanced Version",
                 style={'fontWeight': 'bold', 'marginLeft': '10px'})
    ], style={'display': 'flex', 'alignItems': 'center', 'gap': '10px'}),

    html.Div(html.H1('CS-340 Dashboard'), style={'textAlign': 'center'}),
    html.Hr(),

    # Filter Options
    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'Water Rescue', 'value': 'WATER'},
            {'label': 'Mountain Rescue', 'value': 'MOUNTAIN'},
            {'label': 'Disaster Rescue', 'value': 'DISASTER'},
            {'label': 'Reset', 'value': 'RESET'}
        ],
        value='RESET',
        style={'margin': '10px'}
    ),

    html.Hr(),

    # Data Table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[],
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'padding': '5px'},
        style_header={'backgroundColor': 'lightgrey', 'fontWeight': 'bold'}
    ),

    html.Br(),
    html.Hr(),

    # Graph + Map
    html.Div(
        style={'display': 'flex'},
        children=[
            html.Div(id='graph-id'),
            html.Div(id='map-id')
        ]
    )
])


# -----------------------------------------------------------
# Query Builder
# -----------------------------------------------------------
def get_query(filter_type: str):
    if filter_type == 'WATER':
        return {
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    elif filter_type == 'MOUNTAIN':
        return {
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    elif filter_type == 'DISASTER':
        return {
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }
    return {}  # RESET


# -----------------------------------------------------------
# Callback: Update Table
# -----------------------------------------------------------
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    logging.info(f"Filter selected: {filter_type}")

    try:
        query = get_query(filter_type)
        results = db.read(query)
        logging.info(f"Query returned {len(results)} results.")
    except Exception as e:
        logging.error(f"Database query failed: {e}", exc_info=True)
        return []

    try:
        dff = pd.DataFrame.from_records(results)
        dff.drop(columns=['_id'], inplace=True, errors='ignore')
    except Exception as e:
        logging.error(f"DataFrame conversion failed: {e}", exc_info=True)
        return []

    return dff.to_dict('records')


# -----------------------------------------------------------
# Callback: Update Graph
# -----------------------------------------------------------
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):

    if not viewData:
        logging.warning("Graph update: no data available.")
        fig = px.pie(names=[], values=[], title='Breed Distribution')
        return [dcc.Graph(figure=fig)]

    try:
        dff = pd.DataFrame.from_dict(viewData)
    except Exception as e:
        logging.error(f"Graph data conversion failed: {e}", exc_info=True)
        fig = px.pie(names=[], values=[], title='Breed Distribution')
        return [dcc.Graph(figure=fig)]

    if 'breed' not in dff.columns or dff.empty:
        logging.warning("Breed column missing or empty.")
        fig = px.pie(names=[], values=[], title='Breed Distribution')
        return [dcc.Graph(figure=fig)]

    breed_counts = dff['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']

    fig = px.pie(
        breed_counts,
        names='breed',
        values='count',
        title='Breed Distribution'
    )

    return [dcc.Graph(figure=fig)]


# -----------------------------------------------------------
# Callback: Update Map
# -----------------------------------------------------------
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):

    if not viewData:
        logging.warning("Map update: no data available.")
        return []

    if not index:
        logging.info("Map update: no row selected.")
        return []

    try:
        dff = pd.DataFrame.from_dict(viewData)
        row = index[0]
        lat = dff.iloc[row, 13]
        lon = dff.iloc[row, 14]
    except Exception as e:
        logging.error(f"Map coordinate error: {e}", exc_info=True)
        return []

    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# -----------------------------------------------------------
# Run App
# -----------------------------------------------------------
app.run_server(debug=True)

Dash app running on https://iconboston-granddiego-3000.codio.io/proxy/8050/
